# Train Axiom 500M — Verified Plan (9 fixes)

All 9 review fixes applied. Resumable. Atomic Drive saves. Each step has PASS/FAIL.

In [ ]:
# Step 1: Mount Drive + verify space
from google.colab import drive
drive.mount('/content/drive')
import shutil
usage = shutil.disk_usage('/content/drive/MyDrive')
free_gb = usage.free / 1e9
assert free_gb > 5, f'FAIL: {free_gb:.1f}GB free, need >5GB'
print(f'PASS: Drive mounted, {free_gb:.1f}GB free')

In [ ]:
# Step 2: Verify 2GB Drive write
import torch, os
path = '/content/drive/MyDrive/axiom_drive_test_2gb.pt'
dummy = {'data': torch.randn(256, 1024, 1024)}
torch.save(dummy, path)
loaded = torch.load(path, map_location='cpu')
assert loaded['data'].shape == (256, 1024, 1024)
size_mb = os.path.getsize(path) / 1e6
os.remove(path)
print(f'PASS: wrote and read {size_mb:.0f}MB to Drive')

In [ ]:
# Step 3: Clone + install
!git clone https://github.com/MetaCortex-Dynamics/Axiom-Ref.git
%cd Axiom-Ref
!pip install -q torch tokenizers

In [ ]:
# Step 4: Verify corpus
import json, os
paths = ['corpus/axiom/pairs.json', 'corpus/distilled/self_distilled_pairs.json', 'corpus/teacher/teacher_pairs.json']
total = 0
for p in paths:
    assert os.path.exists(p), f'FAIL: {p} missing'
    with open(p) as f:
        n = len(json.load(f))
    print(f'  {p}: {n}')
    total += n
assert total >= 30000, f'FAIL: only {total} pairs'
print(f'PASS: {total} pairs')

In [ ]:
# Step 5: Verify GPU
import torch
assert torch.cuda.is_available(), 'FAIL: no CUDA'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'PASS: {name}, {vram:.0f}GB')

In [ ]:
# Step 6: Verify script has all 9 fixes
src = open('scripts/train_500m_v2.py').read()
checks = [
    ('os.makedirs', 'makedirs' in src),
    ('full checkpoint (optimizer)', '"optimizer"' in src and '"scheduler"' in src and '"scaler"' in src),
    ('Drive space check', 'disk_usage' in src),
    ('resume logic', 'load_checkpoint' in src),
    ('atomic rename', 'os.replace' in src),
    ('Drive log', 'DRIVE_LOG' in src),
    ('tmp file save', '.tmp.pt' in src),
    ('[SAVE] log', '[SAVE]' in src),
]
all_pass = True
for name, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    if not ok: all_pass = False
    print(f'  [{status}] {name}')
assert all_pass, 'FAIL: script missing fixes'
print('PASS: all fixes verified')

In [ ]:
# Step 7: Train (logs to Drive, checkpoints to Drive, resumable)
!python -u scripts/train_500m_v2.py 2>&1 | tee /content/drive/MyDrive/axiom_500m_train.log

In [ ]:
# Step 8: Verify checkpoint on Drive
import torch
path = '/content/drive/MyDrive/axiom_500m_32k_checkpoint.pt'
ckpt = torch.load(path, map_location='cpu')
print(f'PASS: epoch={ckpt["epoch"]}, best={ckpt["best"]:.4f}, keys={len(ckpt["model"])}')

In [ ]:
# Step 9: Download final model
from google.colab import files
files.download('/content/drive/MyDrive/axiom_500m_decoder_final.pt')